# Buổi 5: Xử lý và biến đổi dữ liệu với Pandas
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Phát hiện và xử lý dữ liệu trùng lặp, thiếu, sai định dạng, ngoại lai
- Chuyển kiểu dữ liệu, xử lý ngày tháng, tạo cột mới
- Tổng hợp bằng groupby, pivot_table; kết hợp bảng bằng merge, concat
- Tính chỉ số RFM cho khách hàng

**Dữ liệu dùng trong buổi:** `ecommerce_orders_raw.csv`, `customers.csv` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [3]:
import numpy as np
import pandas as pd
DATA = "../data/"
pd.set_option("display.max_columns", 30)

## Phần A. Bài tập có hướng dẫn

Làm sạch bộ dữ liệu đơn hàng thương mại điện tử năm 2025 rồi phân tích.

### A1. Khám phá vấn đề của dữ liệu thô

In [6]:
print("Giá trị thiếu mỗi cột:\n", raw.isna().sum())
print("\nSố dòng trùng hoàn toàn:", raw.duplicated().sum())
print("\nCác giá trị cột vùng:", raw["vung"].unique())
print("\nVí dụ đơn giá:", raw["don_gia"].sample(6, random_state=1).tolist())

Giá trị thiếu mỗi cột:
 ma_don         0
ngay_dat       0
ma_kh          0
vung          20
kenh           0
nhom_hang      0
san_pham       0
so_luong      61
don_gia        0
trang_thai     0
dtype: int64

Số dòng trùng hoàn toàn: 45

Các giá trị cột vùng: <StringArray>
[    'Miền Nam',   'Miền Trung',     'Miền Bắc',            nan,
   ' Miền Nam ',   ' Miền Bắc ', ' Miền Trung ',     'MIỀN NAM',
   'MIỀN TRUNG',     'MIỀN BẮC']
Length: 10, dtype: str

Ví dụ đơn giá: ['467000', '287000', '472000', '9109000', '8070000', '263000']


**Nhận xét:** (1) có dòng trùng; (2) `vung` viết không thống nhất và có giá trị thiếu; (3) `so_luong` thiếu;
(4) `don_gia` là chuỗi, có dạng "1.250.000 đ", có giá trị âm và giá trị bất thường; (5) `ngay_dat` lẫn hai định dạng.

### A2. Bỏ trùng lặp

In [7]:
df = raw.drop_duplicates().copy()
print("Trước:", len(raw), "→ Sau:", len(df))
print("Mã đơn còn trùng?", df["ma_don"].duplicated().sum())

Trước: 3816 → Sau: 3771
Mã đơn còn trùng? 0


### A3. Chuẩn hóa chuỗi và kiểu số

In [8]:
df["vung"] = df["vung"].str.strip().str.title()
print(df["vung"].value_counts(dropna=False))

def lam_sach_gia(x):
    if isinstance(x, str):
        x = x.replace("đ", "").replace(".", "").strip()
    return float(x)

df["don_gia"] = df["don_gia"].apply(lam_sach_gia)
df["so_luong"] = pd.to_numeric(df["so_luong"], errors="coerce")
df[["don_gia", "so_luong"]].describe()

vung
Miền Nam      1633
Miền Bắc      1344
Miền Trung     774
NaN             20
Name: count, dtype: int64


,don_gia,so_luong
count,3.771000e+03,3711.000000
mean,4.889065e+06,1.683374
std,3.668363e+07,0.745691
min,-1.000000e+00,1.000000
25%,3.830000e+05,1.000000
50%,5.400000e+05,2.000000
75%,5.633000e+06,2.000000
max,9.999990e+08,3.000000


### A4. Ngày tháng
`pd.to_datetime` với `format="mixed"` và `dayfirst=True` hiểu được cả `2025-03-15` lẫn `15/03/2025`.
Sau đó trích xuất tháng, quý, thứ trong tuần bằng `.dt`.

In [9]:
df["ngay_dat"] = pd.to_datetime(df["ngay_dat"], format="mixed", dayfirst=True)
df["thang"] = df["ngay_dat"].dt.month
df["quy"] = df["ngay_dat"].dt.quarter
df["thu"] = df["ngay_dat"].dt.day_name()
print(df["ngay_dat"].min(), "→", df["ngay_dat"].max())
df[["ngay_dat", "thang", "quy", "thu"]].head()

2025-01-01 00:00:00 → 2025-12-31 00:00:00


,ngay_dat,thang,quy,thu
0,2025-01-24,1,1,Friday
1,2025-12-18,12,4,Thursday
2,2025-05-06,5,2,Tuesday
3,2025-06-03,6,2,Tuesday
4,2025-01-26,1,1,Sunday


### A5. Giá trị thiếu và ngoại lai
Chiến lược phổ biến: **xóa** dòng (khi ít và không quan trọng), **điền** giá trị (trung vị, mode, giá trị từ nguồn khác),
hoặc **giữ** và đánh dấu. Ngoại lai phát hiện bằng quy tắc IQR: ngoài khoảng `[Q1 − 1,5·IQR; Q3 + 1,5·IQR]`
tính **trong từng nhóm hàng** (vì laptop vốn đắt hơn phụ kiện).

In [10]:
# 1) số lượng thiếu → điền bằng trung vị (1)
df["so_luong"] = df["so_luong"].fillna(df["so_luong"].median())

# 2) vùng thiếu → lấy từ bảng khách hàng
kh = pd.read_csv(DATA + "customers.csv")
vung_kh = kh.set_index("ma_kh")["vung"]
df["vung"] = df["vung"].fillna(df["ma_kh"].map(vung_kh))

# 3) đơn giá âm → lỗi nhập liệu, loại bỏ
df = df[df["don_gia"] > 0]

# 4) ngoại lai theo IQR trong từng nhóm hàng
def la_ngoai_lai(g):
    q1, q3 = g.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (g < q1 - 1.5 * iqr) | (g > q3 + 1.5 * iqr)

df["ngoai_lai"] = df.groupby("nhom_hang")["don_gia"].transform(la_ngoai_lai)
print("Số ngoại lai:", df["ngoai_lai"].sum())
print(df[df["ngoai_lai"]][["ma_don", "nhom_hang", "don_gia"]])
df = df[~df["ngoai_lai"]].drop(columns="ngoai_lai")
print("Còn lại:", len(df), "dòng; thiếu:", df.isna().sum().sum())

Số ngoại lai: 5
        ma_don   nhom_hang      don_gia
789   DH003663  Thời trang  999999000.0
863   DH001732    Phụ kiện  999999000.0
1635  DH003565  Điện thoại  999999000.0
2503  DH003296     Mỹ phẩm  999999000.0
3731  DH000032  Thời trang  999999000.0
Còn lại: 3761 dòng; thiếu: 0


### A6. Cột mới, lọc đơn hoàn thành và tổng hợp

In [11]:
df["so_luong"] = df["so_luong"].astype(int)
df["doanh_thu"] = df["so_luong"] * df["don_gia"]
ht = df[df["trang_thai"] == "Hoàn thành"].copy()
print("Tỷ lệ hủy đơn:", round((df["trang_thai"] == "Đã hủy").mean() * 100, 1), "%")

theo_thang = ht.groupby("thang").agg(so_don=("ma_don", "count"),
                                     doanh_thu_ty=("doanh_thu", lambda x: x.sum() / 1e9))
theo_thang.round(2)

Tỷ lệ hủy đơn: 19.8 %


,so_don,doanh_thu_ty
thang,,
1,343,1.76
2,216,1.23
3,235,1.47
4,270,1.71
5,238,1.30
6,226,1.38
7,227,1.44
8,254,1.32
9,228,1.52


`pivot_table` – bảng tổng hợp hai chiều giống PivotTable trong Excel:

In [12]:
pv = pd.pivot_table(ht, index="nhom_hang", columns="kenh", values="doanh_thu",
                    aggfunc="sum", margins=True, margins_name="Tổng") / 1e6
pv.round(0)

kenh,App,Cửa hàng,Sàn TMĐT,Website,Tổng
nhom_hang,,,,,
Gia dụng,183.0,90.0,156.0,170.0,599.0
Laptop,3206.0,1676.0,3100.0,2637.0,10619.0
Mỹ phẩm,107.0,46.0,101.0,77.0,330.0
Phụ kiện,140.0,66.0,142.0,122.0,470.0
Thời trang,134.0,60.0,147.0,125.0,466.0
Điện thoại,1438.0,840.0,1550.0,1606.0,5433.0
Tổng,5207.0,2779.0,5195.0,4736.0,17917.0


`merge` ghép thông tin khách hàng (tuổi, giới tính) vào đơn hàng:

In [13]:
ht = ht.merge(kh[["ma_kh", "tuoi", "gioi_tinh"]], on="ma_kh", how="left")
ht["nhom_tuoi"] = pd.cut(ht["tuoi"], bins=[17, 25, 35, 50, 65], labels=["18-25", "26-35", "36-50", "51-65"])
ht.groupby("nhom_tuoi", observed=True)["doanh_thu"].agg(["count", "mean"]).round(0)

,count,mean
nhom_tuoi,,
18-25,456,6433509.0
26-35,705,6150372.0
36-50,902,5978629.0
51-65,953,5513939.0


### A7. Phân tích RFM
RFM đánh giá giá trị khách hàng bằng 3 chỉ số: **R**ecency – số ngày từ lần mua gần nhất,
**F**requency – số đơn, **M**onetary – tổng chi tiêu. Mỗi chỉ số chấm điểm 1–5 bằng ngũ phân vị (`pd.qcut`).

In [14]:
ngay_moc = ht["ngay_dat"].max() + pd.Timedelta(days=1)
rfm = ht.groupby("ma_kh").agg(
    R=("ngay_dat", lambda x: (ngay_moc - x.max()).days),
    F=("ma_don", "nunique"),
    M=("doanh_thu", "sum"),
)
rfm["R_diem"] = pd.qcut(rfm["R"], 5, labels=[5, 4, 3, 2, 1]).astype(int)          # gần đây → điểm cao
rfm["F_diem"] = pd.qcut(rfm["F"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_diem"] = pd.qcut(rfm["M"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM"] = rfm["R_diem"].astype(str) + rfm["F_diem"].astype(str) + rfm["M_diem"].astype(str)
rfm.sort_values("M", ascending=False).head()

,R,F,M,R_diem,F_diem,M_diem,RFM
ma_kh,,,,,,,
KH0542,18,14,290436000.0,4,5,5,455
KH0475,38,16,239879000.0,3,5,5,355
KH0455,8,12,238994000.0,5,5,5,555
KH0513,15,11,233108000.0,5,5,5,555
KH0020,8,12,221724000.0,5,5,5,555


In [15]:
def phan_khuc(r):
    if r.R_diem >= 4 and r.F_diem >= 4:
        return "Khách hàng tốt nhất"
    if r.F_diem >= 4:
        return "Trung thành"
    if r.R_diem >= 4:
        return "Mới / tiềm năng"
    if r.R_diem <= 2 and r.F_diem >= 3:
        return "Có nguy cơ rời bỏ"
    if r.R_diem <= 2:
        return "Ngủ đông"
    return "Cần chăm sóc"

rfm["phan_khuc"] = rfm.apply(phan_khuc, axis=1)
tong_hop = rfm.groupby("phan_khuc").agg(so_kh=("R", "size"), R_tb=("R", "mean"), F_tb=("F", "mean"), M_tb_trieu=("M", lambda x: x.mean() / 1e6))
tong_hop.round(1).sort_values("M_tb_trieu", ascending=False)

,so_kh,R_tb,F_tb,M_tb_trieu
phan_khuc,,,,
Khách hàng tốt nhất,147,17.4,10.0,78.0
Trung thành,82,66.8,8.6,50.8
Có nguy cơ rời bỏ,54,175.9,3.6,10.3
Mới / tiềm năng,85,16.7,3.0,9.4
Cần chăm sóc,56,51.1,2.7,6.1
Ngủ đông,149,191.0,1.6,3.9


In [16]:
ht.to_csv("don_hang_sach.csv", index=False, encoding="utf-8-sig")
rfm.to_csv("rfm.csv", encoding="utf-8-sig")

## Phần B. Bài tập tự làm

Dùng các DataFrame `df` (đã làm sạch, gồm cả đơn hủy), `ht` (đơn hoàn thành) và `rfm` ở Phần A.

### Bài B1. Báo cáo chất lượng dữ liệu

Viết hàm `kiem_tra(df)` trả về DataFrame có một dòng cho mỗi cột với: kiểu dữ liệu, số giá trị thiếu, % thiếu,
số giá trị khác nhau. Chạy hàm trên `raw` và trên `df` để so sánh trước – sau khi làm sạch.

In [18]:

def kiem_tra(data):
    bao_cao = pd.DataFrame({
        "Kieu_du_lieu": data.dtypes.astype(str),
        "So_gia_tri_thieu": data.isna().sum(),
        "Ty_le_thieu_%": (data.isna().mean() * 100).round(2),
        "So_gia_tri_khac_nhau": data.nunique()
    })
    return bao_cao

print("=== DỮ LIỆU TRƯỚC KHI LÀM SẠCH ===")
display(kiem_tra(raw))

print("\n=== DỮ LIỆU SAU KHI LÀM SẠCH ===")
display(kiem_tra(df))


=== DỮ LIỆU TRƯỚC KHI LÀM SẠCH ===


,Kieu_du_lieu,So_gia_tri_thieu,Ty_le_thieu_%,So_gia_tri_khac_nhau
ma_don,str,0,0.00,3771
ngay_dat,str,0,0.00,511
ma_kh,str,0,0.00,600
vung,str,20,0.52,9
kenh,str,0,0.00,4
nhom_hang,str,0,0.00,6
san_pham,str,0,0.00,30
so_luong,float64,61,1.60,3
don_gia,str,0,0.00,1874
trang_thai,str,0,0.00,2



=== DỮ LIỆU SAU KHI LÀM SẠCH ===


,Kieu_du_lieu,So_gia_tri_thieu,Ty_le_thieu_%,So_gia_tri_khac_nhau
ma_don,str,0,0.0,3761
ngay_dat,datetime64[us],0,0.0,365
ma_kh,str,0,0.0,600
vung,str,0,0.0,3
kenh,str,0,0.0,4
nhom_hang,str,0,0.0,6
san_pham,str,0,0.0,30
so_luong,int64,0,0.0,3
don_gia,float64,0,0.0,1810
trang_thai,str,0,0.0,2


NHẬN XÉT:
- Ban đầu có 3816 dòng, sau làm sạch còn 3761 dòng.
- Số ô thiếu trước: 81
- Số ô thiếu sau: 0
Dữ liệu sau làm sạch đã được chuẩn hóa để phục vụ phân tích.

### Bài B2. Doanh thu theo vùng và quý

Từ `ht`, lập pivot table doanh thu (triệu đồng) với dòng là vùng, cột là quý. Thêm cột tăng trưởng quý 4 so với quý 3 (%).
Vùng nào tăng trưởng mạnh nhất?

In [20]:

# Tổng hợp doanh thu theo vùng và quý
bang_dt = pd.pivot_table(
    ht,
    index="vung",
    columns="quy",
    values="doanh_thu",
    aggfunc="sum",
    fill_value=0
) / 1e6

# Tính tăng trưởng quý 4 so với quý 3
bang_dt["Tang_truong_Q4_Q3_%"] = (
    (bang_dt[4] - bang_dt[3]) / bang_dt[3] * 100
).round(2)

print("DOANH THU THEO VÙNG VÀ QUÝ (TRIỆU ĐỒNG)")
display(bang_dt.round(2))

vung_top = bang_dt["Tang_truong_Q4_Q3_%"].idxmax()

print("Vùng tăng trưởng mạnh nhất:", vung_top)
print(
    "Tăng trưởng:",
    bang_dt.loc[vung_top, "Tang_truong_Q4_Q3_%"],
    "%"
)

DOANH THU THEO VÙNG VÀ QUÝ (TRIỆU ĐỒNG)


quy,1,2,3,4,Tang_truong_Q4_Q3_%
vung,,,,,
Miền Bắc,1511.82,1492.22,1620.80,1605.23,-0.96
Miền Nam,2010.30,2108.86,1782.31,1918.00,7.61
Miền Trung,940.44,792.27,880.26,1254.70,42.54


Vùng tăng trưởng mạnh nhất: Miền Trung
Tăng trưởng: 42.54 %


pivot_table() tạo bảng tổng hợp hai chiều giống PivotTable trong Excel. aggfunc="sum" cộng doanh thu, / 1e6 đổi đơn vị sang triệu đồng, còn idxmax() tìm vùng có tỷ lệ tăng trưởng cao nhất.


### Bài B3. Tỷ lệ hủy đơn theo kênh

Từ `df` (gồm cả đơn hủy), tính tỷ lệ hủy đơn (%) theo từng kênh bán và theo nhóm hàng.
Kênh/nhóm hàng nào có tỷ lệ hủy cao nhất? Gợi ý một hành động cho quản lý.

<details><summary>💡 Gợi ý</summary>

Tạo cột 0/1 cho đơn hủy rồi lấy trung bình theo nhóm = tỷ lệ.

</details>

In [21]:

# Tạo cột đánh dấu đơn hủy
df["huy_don"] = (df["trang_thai"] == "Đã hủy").astype(int)

# Tỷ lệ hủy theo kênh
huy_kenh = df.groupby("kenh").agg(
    tong_don=("ma_don", "count"),
    so_don_huy=("huy_don", "sum"),
    ty_le_huy=("huy_don", "mean")
)

huy_kenh["ty_le_huy"] = huy_kenh["ty_le_huy"] * 100

# Tỷ lệ hủy theo nhóm hàng
huy_nhom = df.groupby("nhom_hang").agg(
    tong_don=("ma_don", "count"),
    so_don_huy=("huy_don", "sum"),
    ty_le_huy=("huy_don", "mean")
)

huy_nhom["ty_le_huy"] = huy_nhom["ty_le_huy"] * 100

print("TỶ LỆ HỦY THEO KÊNH")
display(huy_kenh.sort_values("ty_le_huy", ascending=False).round(2))

print("TỶ LỆ HỦY THEO NHÓM HÀNG")
display(huy_nhom.sort_values("ty_le_huy", ascending=False).round(2))

print("Kênh có tỷ lệ hủy cao nhất:", huy_kenh["ty_le_huy"].idxmax())
print("Nhóm hàng có tỷ lệ hủy cao nhất:", huy_nhom["ty_le_huy"].idxmax())


TỶ LỆ HỦY THEO KÊNH


,tong_don,so_don_huy,ty_le_huy
kenh,,,
App,1141,241,21.12
Cửa hàng,535,104,19.44
Website,979,188,19.20
Sàn TMĐT,1106,212,19.17


TỶ LỆ HỦY THEO NHÓM HÀNG


,tong_don,so_don_huy,ty_le_huy
nhom_hang,,,
Laptop,500,106,21.20
Điện thoại,618,129,20.87
Thời trang,711,139,19.55
Phụ kiện,915,177,19.34
Gia dụng,452,87,19.25
Mỹ phẩm,565,107,18.94


Kênh có tỷ lệ hủy cao nhất: App
Nhóm hàng có tỷ lệ hủy cao nhất: Laptop


(df["trang_thai"] == "Đã hủy") kiểm tra đơn hàng có bị hủy không. .astype(int) chuyển True thành 1 và False thành 0. Khi lấy trung bình bằng mean(), ta nhận được tỷ lệ đơn bị hủy trong mỗi nhóm.

### Bài B4. Top sản phẩm và giỏ hàng trung bình

(a) 10 sản phẩm có doanh thu cao nhất và tỷ trọng cộng dồn (%) của chúng (gợi ý `cumsum`).
(b) Giá trị đơn hàng trung bình (AOV) theo kênh và theo tháng; tháng nào AOV cao nhất?

In [22]:

# (a) Top 10 sản phẩm theo doanh thu
doanh_thu_sp = (
    ht.groupby("san_pham")["doanh_thu"]
    .sum()
    .sort_values(ascending=False)
)

top10 = doanh_thu_sp.head(10).to_frame("doanh_thu")

top10["ty_trong_%"] = (
    top10["doanh_thu"] / doanh_thu_sp.sum() * 100
)

top10["ty_trong_cong_don_%"] = (
    top10["ty_trong_%"].cumsum()
)

print("TOP 10 SẢN PHẨM THEO DOANH THU")
display(top10.round(2))

# (b) Tính giá trị của từng đơn hàng
don_hang = ht.groupby("ma_don", as_index=False).agg(
    kenh=("kenh", "first"),
    thang=("thang", "first"),
    gia_tri_don=("doanh_thu", "sum")
)

# AOV theo kênh
aov_kenh = don_hang.groupby("kenh")["gia_tri_don"].mean()

# AOV theo tháng
aov_thang = don_hang.groupby("thang")["gia_tri_don"].mean()

print("AOV THEO KÊNH")
display(aov_kenh.round(0).to_frame("AOV"))

print("AOV THEO THÁNG")
display(aov_thang.round(0).to_frame("AOV"))

print("Tháng có AOV cao nhất:", aov_thang.idxmax())
print("Kênh có AOV cao nhất:", aov_kenh.idxmax())


TOP 10 SẢN PHẨM THEO DOANH THU


,doanh_thu,ty_trong_%,ty_trong_cong_don_%
san_pham,,,
Laptop C,2.397842e+09,13.38,13.38
Laptop D,2.177400e+09,12.15,25.54
Laptop B,2.138361e+09,11.93,37.47
Laptop E,1.997867e+09,11.15,48.62
Laptop A,1.907419e+09,10.65,59.27
Điện thoại C,1.237933e+09,6.91,66.18
Điện thoại D,1.136647e+09,6.34,72.52
Điện thoại E,1.069865e+09,5.97,78.49
Điện thoại B,1.007986e+09,5.63,84.12


AOV THEO KÊNH


,AOV
kenh,
App,5785779.0
Cửa hàng,6446784.0
Sàn TMĐT,5811128.0
Website,5987719.0


AOV THEO THÁNG


,AOV
thang,
1,5124656.0
2,5703671.0
3,6267264.0
4,6344467.0
5,5453046.0
6,6117292.0
7,6349925.0
8,5207012.0
9,6663864.0


Tháng có AOV cao nhất: 11
Kênh có AOV cao nhất: Cửa hàng


cumsum() tính tỷ trọng cộng dồn của các sản phẩm. Ví dụ, sản phẩm thứ nhất chiếm 10% và sản phẩm thứ hai chiếm 8% thì tỷ trọng cộng dồn đến sản phẩm thứ hai là 18%. Dùng groupby("ma_don") trước khi tính AOV để đảm bảo mỗi đơn hàng chỉ được tính một lần, kể cả khi đơn đó có nhiều dòng sản phẩm.

### Bài B5. Nối dữ liệu RFM với thông tin khách hàng

Ghép `rfm` với `customers.csv` (dùng `merge`). Cho biết với mỗi phân khúc: số khách, tuổi trung bình, tỷ lệ nữ (%)
và vùng có nhiều khách nhất. Khách hàng có nguy cơ rời bỏ tập trung ở vùng nào?

In [23]:

# Đưa mã khách hàng thành cột để ghép bảng
rfm_kh = rfm.reset_index()

# Ghép với bảng khách hàng
rfm_kh = rfm_kh.merge(
    kh[["ma_kh", "tuoi", "gioi_tinh", "vung"]],
    on="ma_kh",
    how="left"
)

# Hàm tìm vùng có nhiều khách nhất
def vung_dong_nhat(x):
    return x.value_counts().idxmax()

# Tổng hợp thông tin theo phân khúc
bao_cao = rfm_kh.groupby("phan_khuc").agg(
    so_khach=("ma_kh", "nunique"),
    tuoi_tb=("tuoi", "mean"),
    ty_le_nu=("gioi_tinh", lambda x: (x == "Nữ").mean() * 100),
    vung_dong_nhat=("vung", vung_dong_nhat)
)

print("THÔNG TIN CÁC PHÂN KHÚC RFM")
display(bao_cao.round(2))

# Phân bố khách có nguy cơ rời bỏ
nguy_co = rfm_kh[
    rfm_kh["phan_khuc"] == "Có nguy cơ rời bỏ"
]

phan_bo_vung = nguy_co["vung"].value_counts()

print("KHÁCH CÓ NGUY CƠ RỜI BỎ THEO VÙNG")
display(phan_bo_vung)

if len(phan_bo_vung) > 0:
    print("Vùng tập trung nhiều nhất:", phan_bo_vung.idxmax())


THÔNG TIN CÁC PHÂN KHÚC RFM


,so_khach,tuoi_tb,ty_le_nu,vung_dong_nhat
phan_khuc,,,,
Có nguy cơ rời bỏ,54,39.70,42.59,Miền Nam
Cần chăm sóc,56,39.73,55.36,Miền Bắc
Khách hàng tốt nhất,147,41.20,52.38,Miền Nam
Mới / tiềm năng,85,41.91,41.18,Miền Nam
Ngủ đông,149,41.97,53.69,Miền Bắc
Trung thành,82,43.17,46.34,Miền Nam


KHÁCH CÓ NGUY CƠ RỜI BỎ THEO VÙNG


vung
Miền Nam      23
Miền Bắc      17
Miền Trung    14
Name: count, dtype: int64

Vùng tập trung nhiều nhất: Miền Nam


## Phần C. Bài tập làm thêm

### Bài C1. Phân tích nhóm khách hàng theo tháng mua đầu tiên (cohort)

Xác định tháng mua đầu tiên của mỗi khách hàng. Lập bảng cohort: dòng = tháng mua đầu, cột = số tháng kể từ lần đầu
(0, 1, 2, ...), giá trị = tỷ lệ khách còn quay lại mua (%). Nhận xét tỷ lệ giữ chân sau 3 tháng.

In [24]:

# Lấy dữ liệu khách hàng và ngày mua
cohort = ht[["ma_kh", "ngay_dat"]].copy()

# Chuyển ngày mua thành tháng
cohort["thang_mua"] = cohort["ngay_dat"].dt.to_period("M")

# Xác định tháng mua đầu tiên của mỗi khách
cohort["thang_dau"] = cohort.groupby("ma_kh")["thang_mua"].transform("min")

# Tính số tháng kể từ lần mua đầu tiên
cohort["so_thang_sau"] = (
    (cohort["thang_mua"].dt.year - cohort["thang_dau"].dt.year) * 12
    + cohort["thang_mua"].dt.month
    - cohort["thang_dau"].dt.month
)

# Đếm số khách quay lại theo từng tháng
bang_cohort = cohort.groupby(
    ["thang_dau", "so_thang_sau"]
)["ma_kh"].nunique().unstack(fill_value=0)

# Tính tỷ lệ giữ chân khách hàng
quy_mo = bang_cohort[0]
ty_le_giu_chan = bang_cohort.div(quy_mo, axis=0) * 100

print("BẢNG COHORT - TỶ LỆ GIỮ CHÂN (%)")
display(ty_le_giu_chan.round(2))

# Chỉ tính những cohort đã đủ 3 tháng theo dõi
thang_cuoi = cohort["thang_mua"].max()
du_3_thang = ty_le_giu_chan.index <= thang_cuoi - 3

if 3 in ty_le_giu_chan.columns:
    giu_chan_3 = ty_le_giu_chan.loc[du_3_thang, 3].mean()
    print("Tỷ lệ giữ chân trung bình sau 3 tháng:",
          round(giu_chan_3, 2), "%")


BẢNG COHORT - TỶ LỆ GIỮ CHÂN (%)


so_thang_sau,0,1,2,3,4,5,6,7,8,9,10,11
thang_dau,,,,,,,,,,,,
2025-01,100.0,38.55,39.36,42.57,36.55,34.14,33.73,36.95,35.74,36.55,43.78,36.55
2025-02,100.0,32.05,39.74,42.31,35.90,30.77,46.15,24.36,30.77,41.03,39.74,0.00
2025-03,100.0,31.67,28.33,33.33,30.00,35.00,28.33,35.00,33.33,38.33,0.00,0.00
2025-04,100.0,28.57,26.19,23.81,21.43,35.71,35.71,26.19,33.33,0.00,0.00,0.00
2025-05,100.0,24.00,24.00,40.00,16.00,48.00,28.00,36.00,0.00,0.00,0.00,0.00
2025-06,100.0,21.05,15.79,26.32,15.79,10.53,26.32,0.00,0.00,0.00,0.00,0.00
2025-07,100.0,15.38,38.46,23.08,15.38,7.69,0.00,0.00,0.00,0.00,0.00,0.00
2025-08,100.0,16.67,8.33,8.33,25.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2025-09,100.0,5.56,5.56,11.11,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Tỷ lệ giữ chân trung bình sau 3 tháng: 27.87 %


to_period("M") chuyển ngày mua thành tháng, ví dụ 15/03/2025 thành 2025-03. transform("min") tìm tháng mua đầu tiên của từng khách. nunique() đếm khách không trùng, còn unstack() chuyển dữ liệu thành bảng có các cột tháng 0, 1, 2, 3… Trong đó tháng 0 là tháng mua đầu tiên, tháng 3 là ba tháng sau lần mua đầu.

### Bài C2. Doanh thu theo ngày trong tuần và trung bình trượt

(a) Doanh thu trung bình theo thứ trong tuần (sắp xếp Thứ 2 → Chủ nhật, đổi tên sang tiếng Việt).
(b) Tạo chuỗi doanh thu theo ngày (`resample("D")`), tính trung bình trượt 7 ngày và tìm 3 tuần có doanh thu cao nhất.

In [27]:

# Tính tổng doanh thu theo ngày
doanh_thu_ngay = (
    ht.set_index("ngay_dat")["doanh_thu"]
    .resample("D")
    .sum()
)

# (a) Doanh thu trung bình theo thứ
tb_theo_thu = doanh_thu_ngay.groupby(
    doanh_thu_ngay.index.day_of_week
).mean()

ten_thu = {
    0: "Thứ 2",
    1: "Thứ 3",
    2: "Thứ 4",
    3: "Thứ 5",
    4: "Thứ 6",
    5: "Thứ 7",
    6: "Chủ nhật"
}

tb_theo_thu = tb_theo_thu.reindex(range(7))
tb_theo_thu.index = [ten_thu[i] for i in range(7)]

print("DOANH THU TRUNG BÌNH THEO THỨ")
display(tb_theo_thu.round(0).to_frame("doanh_thu_tb"))

# (b) Trung bình trượt 7 ngày
trung_binh_truot = doanh_thu_ngay.rolling(
    window=7, min_periods=7
).mean()

bang_truot = pd.DataFrame({
    "doanh_thu_ngay": doanh_thu_ngay,
    "trung_binh_truot_7_ngay": trung_binh_truot
})

print("DOANH THU VÀ TRUNG BÌNH TRƯỢT 7 NGÀY")
display(bang_truot.tail(10).round(0))

# Tính doanh thu theo tuần
doanh_thu_tuan = doanh_thu_ngay.resample("W-SUN").sum()

# Chỉ lấy các tuần có đủ 7 ngày
ngay_dau = doanh_thu_ngay.index.min()
ngay_cuoi = doanh_thu_ngay.index.max()

tuan_day_du = doanh_thu_tuan[
    (doanh_thu_tuan.index - pd.Timedelta(days=6) >= ngay_dau)
    & (doanh_thu_tuan.index <= ngay_cuoi)
]

top3_tuan = tuan_day_du.nlargest(3)

print("TOP 3 TUẦN CÓ DOANH THU CAO NHẤT")
display(top3_tuan.to_frame("doanh_thu_tuan"))

print("Thứ có doanh thu trung bình cao nhất:",
      tb_theo_thu.idxmax())


DOANH THU TRUNG BÌNH THEO THỨ


,doanh_thu_tb
Thứ 2,39673135.0
Thứ 3,51201673.0
Thứ 4,58097434.0
Thứ 5,49272481.0
Thứ 6,47716288.0
Thứ 7,51039308.0
Chủ nhật,46443942.0


DOANH THU VÀ TRUNG BÌNH TRƯỢT 7 NGÀY


,doanh_thu_ngay,trung_binh_truot_7_ngay
ngay_dat,,
2025-12-22,33328000.0,61181714.0
2025-12-23,54727000.0,61996571.0
2025-12-24,110229000.0,61275143.0
2025-12-25,33480000.0,49280429.0
2025-12-26,42455000.0,47739000.0
2025-12-27,75194000.0,51124714.0
2025-12-28,18758000.0,52595857.0
2025-12-29,78898000.0,59105857.0
2025-12-30,12018000.0,53004571.0


TOP 3 TUẦN CÓ DOANH THU CAO NHẤT


,doanh_thu_tuan
ngay_dat,
2025-11-16,589552000.0
2025-01-26,531992000.0
2025-01-19,482114000.0


Thứ có doanh thu trung bình cao nhất: Thứ 4


### Bài C3. Đóng gói quy trình làm sạch

Viết hàm `lam_sach(duong_dan_raw, duong_dan_kh)` gom toàn bộ các bước A2–A6 thành một hàm, trả về DataFrame sạch.
Kiểm tra: kết quả của hàm phải có cùng số dòng với `df` ở Phần A. Vì sao đóng gói thành hàm lại quan trọng
khi dữ liệu được cập nhật hằng tuần?

In [29]:

def lam_sach(duong_dan_raw, duong_dan_kh):
    raw_moi = pd.read_csv(duong_dan_raw)
    kh_moi = pd.read_csv(duong_dan_kh)

    # A2: Bỏ dòng trùng
    sach = raw_moi.drop_duplicates().copy()

    # A3: Chuẩn hóa vùng và kiểu số
    sach["vung"] = sach["vung"].str.strip().str.title()

    def doi_gia(x):
        if isinstance(x, str):
            x = x.replace("đ", "").replace(".", "").strip()
        return float(x)

    sach["don_gia"] = sach["don_gia"].apply(doi_gia)
    sach["so_luong"] = pd.to_numeric(
        sach["so_luong"], errors="coerce"
    )

    # A4: Xử lý ngày tháng
    sach["ngay_dat"] = pd.to_datetime(
        sach["ngay_dat"], format="mixed", dayfirst=True
    )

    sach["thang"] = sach["ngay_dat"].dt.month
    sach["quy"] = sach["ngay_dat"].dt.quarter
    sach["thu"] = sach["ngay_dat"].dt.day_name()

    # A5: Xử lý giá trị thiếu
    sach["so_luong"] = sach["so_luong"].fillna(
        sach["so_luong"].median()
    )

    vung_kh_moi = kh_moi.set_index("ma_kh")["vung"]

    sach["vung"] = sach["vung"].fillna(
        sach["ma_kh"].map(vung_kh_moi)
    )

    # Loại đơn giá không hợp lệ
    sach = sach[sach["don_gia"] > 0].copy()

    # Phát hiện ngoại lai theo từng nhóm hàng
    def la_ngoai_lai_moi(g):
        q1, q3 = g.quantile([0.25, 0.75])
        iqr = q3 - q1
        return (
            (g < q1 - 1.5 * iqr)
            | (g > q3 + 1.5 * iqr)
        )

    sach["ngoai_lai"] = sach.groupby(
        "nhom_hang"
    )["don_gia"].transform(la_ngoai_lai_moi)

    sach = sach[~sach["ngoai_lai"]].drop(
        columns="ngoai_lai"
    ).copy()

    # A6: Tạo cột doanh thu
    sach["so_luong"] = sach["so_luong"].astype(int)
    sach["doanh_thu"] = sach["so_luong"] * sach["don_gia"]

    return sach


# Chạy thử hàm
ket_qua = lam_sach(
    DATA + "ecommerce_orders_raw.csv",
    DATA + "customers.csv"
)

print("Số dòng phần A:", len(df))
print("Số dòng từ hàm:", len(ket_qua))

if len(df) == len(ket_qua):
    print("Kết quả khớp về số dòng!")
else:
    print("Kết quả chưa khớp, cần kiểm tra lại.")


Số dòng phần A: 3761
Số dòng từ hàm: 3761
Kết quả khớp về số dòng!


def lam_sach(...) tạo một hàm gom các bước làm sạch từ A2 đến A6. Hai tham số đầu vào là đường dẫn file đơn hàng và file khách hàng. return sach trả về bảng dữ liệu sau khi xử lý. Khi gọi hàm, Python sẽ thực hiện lại toàn bộ quy trình mà không cần viết từng bước riêng lẻ.